In [9]:
import urllib.request
import zipfile
from pathlib import Path

base = "https://object.pouta.csc.fi/OPUS-Books/v1/moses/"
url = base + "en-sv.txt.zip"

Path("data").mkdir(exist_ok=True)

zip_path = "data/en-sv.txt.zip"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path) as z:
    print("Files:", z.namelist())
    z.extractall("data")

print("Dataset downloaded!")

Files: ['Books.en-sv.en', 'Books.en-sv.sv', 'Books.en-sv.ids', 'README']
Dataset downloaded!


In [11]:
import glob
import random
import json

en_file = glob.glob("data/Books*.en")
sv_file = glob.glob("data/Books*.sv")

print("English files:", en_file)
print("Swedish files:", sv_file)

with open(en_file[0], encoding="utf-8") as f:
    english = [line.strip() for line in f]

with open(sv_file[0], encoding="utf-8") as f:
    swedish = [line.strip() for line in f]

pairs = [
    (en, sv)
    for en, sv in zip(english, swedish)
    if 20 <= len(en) <= 300 and 20 <= len(sv) <= 300
]

pairs = list(dict.fromkeys(pairs))
random.Random(42).shuffle(pairs)

assert len(pairs) >= 1200, f"Only {len(pairs)} usable pairs"

group1 = pairs[:500]
group2 = pairs[500:1000]
test_pairs = pairs[1000:1200]

datasets = {
    "EN_1": [en for en, sv in group1],
    "EN_2": [en for en, sv in group2],
    "SV_1": [sv for en, sv in group1],
    "SV_2": [sv for en, sv in group2]
}

test_data = {
    "EN": [en for en, sv in test_pairs],
    "SV": [sv for en, sv in test_pairs]
}

for name, texts in datasets.items():
    print(name, len(texts), "examples")

with open("data/training_datasets.json", "w", encoding="utf-8") as f:
    json.dump(datasets, f, ensure_ascii=False, indent=2)

with open("data/test_datasets.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

English files: ['data/Books.en-sv.en']
Swedish files: ['data/Books.en-sv.sv']
EN_1 500 examples
EN_2 500 examples
SV_1 500 examples
SV_2 500 examples


In [13]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors
from transformers import PreTrainedTokenizerFast
from pathlib import Path

Path("models").mkdir(exist_ok=True)

special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

backend = Tokenizer(models.WordPiece(unk_token="[UNK]"))
backend.pre_tokenizer = pre_tokenizers.BertPreTokenizer()

trainer = trainers.WordPieceTrainer(
    vocab_size=3000,
    special_tokens=special_tokens
)

all_texts = [
    text
    for texts in datasets.values()
    for text in texts
]

backend.train_from_iterator(all_texts, trainer=trainer)

backend.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[
        ("[CLS]", backend.token_to_id("[CLS]")),
        ("[SEP]", backend.token_to_id("[SEP]"))
    ]
)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend,
    pad_token="[PAD]",
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

print("Vocabulary size:", len(tokenizer))
print(tokenizer.tokenize("Du behöver betala avgiften."))




Vocabulary size: 3000
['D', '##u', 'behöver', 'betala', 'av', '##g', '##ift', '##en', '.']


In [15]:
import torch
import numpy as np
import random
from transformers import BertConfig, BertForMaskedLM

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

config = BertConfig(
    vocab_size=len(tokenizer),
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    intermediate_size=256,
    max_position_embeddings=128
)

initial_model = BertForMaskedLM(config)

initial_model.save_pretrained(
    "models/INITIAL",
    safe_serialization=True
)
tokenizer.save_pretrained("models/INITIAL")

print("Initial model saved.")
print("Parameters:", sum(p.numel() for p in initial_model.parameters()))

del initial_model

W1119 22:19:42.151000 22888 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Initial model saved.
Parameters: 685624


In [19]:
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForLanguageModeling
import gc

class TextDataset(Dataset):
    def __init__(self, texts):
        self.examples = [
            tokenizer(
                text,
                truncation=True,
                max_length=96
            )
            for text in texts
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def train_model(name, texts, steps=100):

    # Identical random seed for every run
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # Load identical initial weights
    model = BertForMaskedLM.from_pretrained("models/INITIAL")
    model.train()

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    loader = DataLoader(
        TextDataset(texts),
        batch_size=8,
        shuffle=True,
        collate_fn=collator
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-4
    )

    iterator = iter(loader)
    losses = []
    completed_steps = 0

    while completed_steps < steps:
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)

        # Skip batches without masked tokens
        if (batch["labels"] != -100).sum().item() == 0:
            continue

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        completed_steps += 1
        losses.append(loss.item())

        if completed_steps % 20 == 0:
            print(
                name,
                "Step:", completed_steps,
                "Loss:", round(loss.item(), 4)
            )

    # Save trained weights, architecture, tokenizer
    path = f"models/{name}"

    model.save_pretrained(path, safe_serialization=True)
    tokenizer.save_pretrained(path)

    print(name, "saved successfully")

    del model, optimizer, loader
    gc.collect()

    return losses

In [21]:
training_history = {}

for name, texts in datasets.items():
    print("\nTraining:", name)

    training_history[name] = train_model(
        name,
        texts,
        steps=100
    )


Training: EN_1


Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

EN_1 Step: 20 Loss: 7.6023
EN_1 Step: 40 Loss: 7.138
EN_1 Step: 60 Loss: 6.5415
EN_1 Step: 80 Loss: 6.7275
EN_1 Step: 100 Loss: 6.7231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

EN_1 saved successfully

Training: EN_2


Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

... (7 outputs left out)

Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

SV_2 Step: 20 Loss: 7.5095
SV_2 Step: 40 Loss: 7.6518
SV_2 Step: 60 Loss: 6.7117
SV_2 Step: 80 Loss: 6.5022
SV_2 Step: 100 Loss: 6.7472


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SV_2 saved successfully


In [36]:
%run pretrained_bert_models/apply_bert_swedish_edits.py

Editing english_bert_uncased from pretrained_bert_models/english_bert_uncased/weights_swedish_bert_uncased.pt
Saved bert_swedish_weight_edits/english_bert_uncased_edited.pt (418 MiB)
Editing english_bert_cased from pretrained_bert_models/english_bert_cased/weights_english_bert_cased.pt
Saved bert_swedish_weight_edits/english_bert_cased_edited.pt (413 MiB)
Editing swedish_bert_base from pretrained_bert_models/swedish_bert_base/swedish_bert_base_weights.pt
Saved bert_swedish_weight_edits/swedish_bert_base_edited.pt (476 MiB)
Editing swedish_bert_new from pretrained_bert_models/swedish_bert_new/model_swedish_bert_new.safetensors
Saved bert_swedish_weight_edits/swedish_bert_new_edited.pt (516 MiB)
Complete. Original model files were not changed.


In [40]:
from pathlib import Path
from zipfile import ZipFile

print("Jupyter working directory:", Path.cwd())

# Look for the downloaded ZIP
locations = [
    Path.cwd(),
    Path.home() / "Downloads"
]

zip_files = [
    p
    for folder in locations
    for p in folder.glob("bert_evaluation_starter*.zip")
]

if not zip_files:
    raise FileNotFoundError(
        "Download bert_evaluation_starter.zip first."
    )

zip_path = zip_files[0]
print("Found ZIP:", zip_path)

# Extract into Jupyter's working directory
with ZipFile(zip_path) as z:
    z.extractall(Path.cwd())

print("Script found:",
      Path("evaluate_bert_before_after.py").exists())
print("Patch found:",
      Path("edit_patch.json").exists())

Jupyter working directory: /Users/hsiar/Documents/selma
Found ZIP: /Users/hsiar/Documents/selma/bert_evaluation_starter.zip
Script found: True
Patch found: True


In [42]:
%run evaluate_bert_before_after.py --n-test 20 --batch-size 4


=== english_bert_uncased ===
  Using ORIGINAL checkpoint: weights_swedish_bert_uncased.pt
  swedish_function (20 words): word acc 0.0% -> 5.0%, loss 5.437 -> 4.568, random loss 5.441
  swedish_content (20 words): word acc 0.0% -> 0.0%, loss 8.664 -> 9.200, random loss 8.667
  english_regression (20 words): word acc 25.0% -> 25.0%, loss 7.240 -> 7.247, random loss 7.241

=== english_bert_cased ===
  Using ORIGINAL checkpoint: weights_english_bert_cased.pt
  swedish_function (20 words): word acc 5.0% -> 5.0%, loss 6.328 -> 5.697, random loss 6.329
  swedish_content (20 words): word acc 0.0% -> 0.0%, loss 7.992 -> 8.827, random loss 7.993
  english_regression (20 words): word acc 35.0% -> 35.0%, loss 6.351 -> 6.389, random loss 6.353

=== swedish_bert_base ===
  Using ORIGINAL checkpoint: swedish_bert_base_weights.pt
  swedish_function (20 words): word acc 95.0% -> 100.0%, loss 0.173 -> 0.047, random loss 0.173
  swedish_content (20 words): word acc 35.0% -> 30.0%, loss 3.643 -> 3.751, r

In [44]:
%run evaluate_bert_before_after.py --n-test 100 --batch-size 4


=== english_bert_uncased ===
  Using ORIGINAL checkpoint: weights_swedish_bert_uncased.pt
  swedish_function (100 words): word acc 0.0% -> 1.0%, loss 5.505 -> 4.859, random loss 5.509
  swedish_content (100 words): word acc 0.0% -> 0.0%, loss 8.825 -> 9.356, random loss 8.829
  english_regression (100 words): word acc 28.0% -> 28.0%, loss 6.256 -> 6.258, random loss 6.257

=== english_bert_cased ===
  Using ORIGINAL checkpoint: weights_english_bert_cased.pt
  swedish_function (100 words): word acc 2.0% -> 3.0%, loss 6.082 -> 5.594, random loss 6.083
  swedish_content (100 words): word acc 0.0% -> 0.0%, loss 7.977 -> 8.813, random loss 7.978
  english_regression (100 words): word acc 33.0% -> 33.0%, loss 6.097 -> 6.098, random loss 6.099

=== swedish_bert_base ===
  Using ORIGINAL checkpoint: swedish_bert_base_weights.pt
  swedish_function (100 words): word acc 90.0% -> 93.0%, loss 0.364 -> 0.234, random loss 0.364
  swedish_content (100 words): word acc 41.0% -> 38.0%, loss 2.806 -> 2

In [46]:
import pandas as pd

df = pd.read_csv("bert_evaluation/results.csv")
display(df.round(4))

,model,suite,n_words,original_word_acc,edited_word_acc,change_word_acc_pp,random_mean_word_acc,original_token_loss,edited_token_loss,loss_improvement,random_mean_token_loss
0,english_bert_uncased,swedish_function,100,0.00,0.01,1.0,0.00,5.5053,4.8595,0.6458,5.5090
1,english_bert_uncased,swedish_content,100,0.00,0.00,0.0,0.00,8.8247,9.3557,-0.5310,8.8289
2,english_bert_uncased,english_regression,100,0.28,0.28,0.0,0.28,6.2565,6.2584,-0.0020,6.2572
3,english_bert_cased,swedish_function,100,0.02,0.03,1.0,0.02,6.0822,5.5938,0.4884,6.0831
4,english_bert_cased,swedish_content,100,0.00,0.00,0.0,0.00,7.9765,8.8134,-0.8369,7.9777
5,english_bert_cased,english_regression,100,0.33,0.33,0.0,0.33,6.0974,6.0979,-0.0005,6.0986
6,swedish_bert_base,swedish_function,100,0.90,0.93,3.0,0.90,0.3644,0.2342,0.1301,0.3644
7,swedish_bert_base,swedish_content,100,0.41,0.38,-3.0,0.41,2.8058,2.9081,-0.1023,2.8061
8,swedish_bert_base,english_regression,100,0.08,0.08,0.0,0.08,5.8677,5.8854,-0.0177,5.8683
9,swedish_bert_new,swedish_function,100,0.80,0.87,7.0,0.80,0.7934,0.5337,0.2597,0.7934
